<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Unsupervised Learning - DBSCAN - Network-Intrusion-Detektor
</b></font> </br></p>

---


# 0  | Install & Import
***

In [ ]:
# Install

In [ ]:
# Daten
from pandas import read_csv, DataFrame, concat

# Dimensionality Reduction
from sklearn.decomposition import PCA

# Skalierung
from sklearn.preprocessing import StandardScaler

# Modell (Clustering)
from sklearn.cluster import DBSCAN

# Modell (Klassifikation / Split)
from sklearn.model_selection import train_test_split

# Evaluation
from sklearn.metrics import (
    silhouette_score,          # Clustering
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)

# Visualisierung
import plotly.express as px
import plotly.subplots as sp
import matplotlib.pyplot as plt

In [ ]:
# Warnung ausstellen
import warnings
warnings.filterwarnings("ignore")

# 1  | Understand
***

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>

<p><font color='black' size="5">
Anwendungsfall
</font></p>

Dies ist der Datensatz, der für den Third International Knowledge Discovery and Data Mining Tools Competition verwendet wurde. Die Wettbewerbsaufgabe bestand darin, einen Netzwerk-Intrusion-Detektor zu bauen, ein Vorhersagemodell, das in der Lage ist, zwischen schlechten Verbindungen, sogenannten Intrusionen oder Angriffen, und guten normalen Verbindungen zu unterscheiden. Diese Datenbank enthält einen zu prüfenden Standarddatensatz, der eine Vielzahl von Eindringversuchen umfasst, die in einer militärischen Netzwerkumgebung simuliert wurden.

Der ursprüngliche KDD Cup 1999-Datensatz aus dem  UCI-Repositorium für maschinelles Lernen  enthält 41 Attribute (34 kontinuierlich und 7 kategorial), sie werden jedoch auf 3 Attribute (Dauer, src_bytes, dst_bytes) reduziert.

Da die kontinuierlichen Attributwerte um '0' herum konzentriert sind, haben wir jeden Wert durch y = log(x + 0,1) in einen Wert weit von '0' transformiert.

Aus KDD Cup 1999-Datensatz werden ca. 500k „http“-Dienstdaten verwendet.

Der KDD Cup ist der jährliche Data-Mining- und Knowledge-Discovery-Wettbewerb, der von der ACM Special Interest Group on Knowledge Discovery and Data Mining, der führenden Berufsorganisation von Data-Minern, organisiert wird.


[DataSet](https://www.openml.org/search?type=data&status=active&id=1113)

[Info](http://odds.cs.stonybrook.edu/http-kddcup99-dataset/)


[UCI](https://archive.ics.uci.edu/ml/datasets/kdd+cup+1999+data)

[KDD](https://kdd.org/kdd-cup)



**Datensatz:**

| feature name  | description                                      | type        |
|------------|--------------------------------------------------|-------------|
| duration   | length (number of seconds) of the connection     | continuous  |
| src_bytes  | number of data bytes from source to destination  | continuous  |
| dst_bytes  | number of data bytes from destination to source  | continuous  |

<br>
<br>


In [ ]:
df = read_csv(
    "https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/kddcup1999_xs.csv"
)

In [ ]:
data = df.copy()
target = data.pop("Intrusion")

In [ ]:
target.value_counts()

# 2 | Prepare

---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Datentyp ermitteln/ändern</br>
✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>

 <p><font color='black' size="5">
Train-Test-Split (nur zur Reduktion des Datenvolumens)
</font></p>

Erfolgt nur, um das Datenvolumen zu reduzieren. Bei 100k Datensätze erfolgt Abbruch mit Hinweis auf Bezahlversion. 😉

In [ ]:
data, _, target, _ = train_test_split(data, target, train_size=0.5, random_state=42, stratify=target)
data.shape, target.shape

<p><font color='black' size="5">
Skalierung
</font></p>

DBSCAN und PCA vergleichen Abstände zwischen Datenpunkten. Deshalb werden die numerischen Features vor beiden Verfahren standardisiert. Der Parameter `eps` bezieht sich danach auf die skalierte Datenbasis.

In [ ]:
scaler = StandardScaler()
data_scaled = DataFrame(
    scaler.fit_transform(data), columns=data.columns, index=data.index
)


In [ ]:
target.value_counts()

# 3 | Modeling
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Cross-Validiation</br>
✅ Bootstrapping</br>
✅ Regularization</br>

<p><font color='black' size="5">🔵 Algorithmus: DBSCAN (Dichtebasiertes Clustering)</font></p>

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) erkennt Cluster anhand der lokalen Datendichte. Im Gegensatz zu K-Means muss die Anzahl der Cluster nicht vorab angegeben werden – das Modell findet sie selbst. Datenpunkte in dünn besiedelten Bereichen werden als Rauschen (Noise) markiert und keinem Cluster zugeordnet.

**Analogie:** In einer Stadt bilden dicht besiedelte Stadtteile natürliche Cluster. Vereinzelte Häuser auf dem Land entsprechen dem Rauschen – keiner Gruppe zugehörig.

**Wichtige Parameter:**

| Parameter | Bedeutung |
|-----------|----------|
| `eps` | Maximaler Abstand, um als Nachbar zu gelten |
| `min_samples` | Mindestanzahl Nachbarn für einen Kernpunkt |
| Kernpunkt | Punkt mit mind. `min_samples` Nachbarn im Radius `eps` |
| Randpunkt | Im Radius eines Kernpunkts, aber selbst kein Kernpunkt |
| Rauschen | Weder Kern- noch Randpunkt – keinem Cluster zugeordnet |

**In der Praxis relevant wenn:**
- Cluster unregelmäßige Formen oder sehr unterschiedliche Dichten haben (K-Means würde versagen)
- Ausreißer automatisch identifiziert werden sollen — DBSCAN markiert sie als Noise-Punkte
- Die Anzahl der Cluster nicht vorab bekannt ist

**Nicht geeignet wenn:**
- Die Daten hochdimensional sind → Distanzmaße verlieren an Aussagekraft (Fluch der Dimensionalität)
- Alle Cluster ähnlich kompakt sind → dann ist K-Means einfacher und schneller

**Typischer Fehler:**
`eps` ohne datengestützte Analyse wählen. Einen k-Distanz-Plot nutzen, um den optimalen `eps`-Wert zu identifizieren — falsch gewähltes `eps` ergibt entweder einen einzigen Cluster oder nur Rauschen.

In [ ]:
#@markdown   <p><font size="4" color='green'>  DBSCAN – Clustering-Prinzip</font> </br></p>

import base64
from IPython.display import Image, display

diagram = """
flowchart TD
    DATA[Datenpunkte] --> CHECK{Nachbarn im Radius eps?}
    CHECK -->|ja, groesser gleich min_samples| CORE[Kernpunkt]
    CHECK -->|ja, kleiner min_samples| BORDER[Randpunkt]
    CHECK -->|nein| NOISE[Rauschen - kein Cluster]
    CORE --> CLUSTER[Cluster wächst]
    BORDER --> CLUSTER
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=450))

 <p><font color='black' size="5">
Modellauswahl & Training
</font></p>

In [ ]:
model = DBSCAN(eps=0.6, min_samples=30)

In [ ]:
model.fit(data_scaled)

In [ ]:
DataFrame({"Cluster": model.labels_}).value_counts()

# 4 | Evaluate
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Robustheitstest erstellen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>

<p><font color='black' size="5">
Silhouette Koeffizient
</font></p>

In [ ]:
mask = model.labels_ != -1
silhouette_coef = silhouette_score(data_scaled[mask], model.labels_[mask])
print("Silhouette-Koeffizient:", silhouette_coef)

<p><font color='black' size="5">
UmKodierung Ergebnisse DBSCAN
</font></p>

In [ ]:
target_pred = DataFrame(model.labels_).copy()
target_pred.columns = ["Intrusion"]
target_pred.replace([-1, 0, 1, 2], [0, 0, 0, 1], inplace=True)

In [ ]:
target_pred.value_counts()


<p><font color='black' size="5">
Confusion Matrix
</font></p>

In [ ]:
conf_matrix = confusion_matrix(target, target_pred["Intrusion"])
plt.rcParams["figure.figsize"] = [5, 5]
disp = ConfusionMatrixDisplay(conf_matrix, display_labels=["No", "Yes"])
disp.plot(cmap="Blues")

In [ ]:
print(classification_report(target, target_pred["Intrusion"], target_names=["No", "Yes"]))

<p><font color='black' size="5">
Aufbau Analysewürfel
</font></p>

In [ ]:
# Übernahme der Testdaten
cube = data.copy()
cube.reset_index(inplace=True)

# Übernahme Target real & predict
cube["real"] = target.values
cube["predict"] = target_pred["Intrusion"].values

In [ ]:
# Erstellung 2D Features über Dimensionsreduktion PCA - unsupervised
pca = PCA(n_components=2)
pca = pca.fit_transform(data_scaled)
pca_df = DataFrame(pca)

# Cube um pca erweitern
cube["PCA1"] = pca_df[0]
cube["PCA2"] = pca_df[1]

<p><font color='black' size="5">
Visualisierung real vs predict
</font></p>

In [ ]:
# Histogramm
title_ = "Histogramm real vs predict"
fig = px.histogram(cube, x=["real", "predict"], nbins=2, text_auto=".2s", title=title_)
fig.update_layout(barmode="group", bargap=0.1, width=600, height=600)
fig.show()

In [ ]:
# 2 x Scatterplots

cube["real_cat"] = cube["real"].astype(str)
cube["predict_cat"] = cube["predict"].astype(str)

title_ = "Streupunktdiagramm real"
img1 = px.scatter(cube, x="PCA1", y="PCA2", color="real_cat", width=600, height=600)

title_ = "Streupunktdiagramm predict"
img2 = px.scatter(cube, x="PCA1", y="PCA2", color="predict_cat", width=600, height=600)

fig = sp.make_subplots(
    rows=1, cols=2, subplot_titles=("Scatterplot real", "Scatterplot predict")
)

for trace in img1.data:
    fig.add_trace(trace, row=1, col=1)
for trace in img2.data:
    fig.add_trace(trace, row=1, col=2)

# Layout anpassen
fig.update_layout(width=1000, height=500, title_text=title_)

# Plot anzeigen
fig.show()

In [ ]:
# real <> predict
cube[cube.real != cube.predict].describe().T

In [ ]:
cube[cube.real != cube.predict]

# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>